In [1]:
from pathlib import Path

import pandas as pd


# Detect project root automatically

CURRENT_FOLDER = Path.cwd()
PROJECT_ROOT = None


for folder in [
    CURRENT_FOLDER,
    *CURRENT_FOLDER.parents
]:

    candidate = (
        folder
        / "data"
        / "CAS_Data_public.csv"
    )

    if candidate.exists():

        PROJECT_ROOT = folder
        break


if PROJECT_ROOT is None:

    raise FileNotFoundError(
        "Could not find the project root containing "
        "data/CAS_Data_public.csv."
    )


DATA_DIR = (
    PROJECT_ROOT
    / "data"
)


CAS_PATH = (
    DATA_DIR
    / "CAS_Data_public.csv"
)


JUNCTION_PATH = (
    DATA_DIR
    / "spatial"
    / "CAS_Junction_enriched.csv"
)


NZDEP_PATH = (
    DATA_DIR
    / "nzdep2023"
    / "CAS_NZDep2023_enriched.csv"
)


LCDB_PATH = (
    DATA_DIR
    / "lcdb"
    / "CAS_LCDB_v6_enriched.csv"
)


TERRAIN_PATH = (
    DATA_DIR
    / "terrain"
    / "CAS_LINZ_terrain_enriched.csv"
)


paths = {
    "CAS":
        CAS_PATH,

    "Junction":
        JUNCTION_PATH,

    "NZDep2023":
        NZDEP_PATH,

    "LCDB":
        LCDB_PATH,

    "Terrain":
        TERRAIN_PATH
}


print(
    "Combined National Context Source Files"
)


for name, path in paths.items():

    print(
        f"\n{name}:"
    )

    print(
        path
    )

    print(
        "Exists:",
        path.exists()
    )


missing_files = [
    name
    for name, path in paths.items()
    if not path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Missing required source files: "
        + ", ".join(
            missing_files
        )
    )


# Read only headers first

required_feature_columns = {
    "Junction":
        "dist_to_junction_m",

    "NZDep2023":
        "NZDep2023_Score",

    "LCDB":
        "LCDB_Group_clean",

    "Terrain":
        "Elevation_m"
}


print(
    "\nRequired feature checks:"
)


for name, feature in (
    required_feature_columns.items()
):

    columns = (
        pd.read_csv(
            paths[
                name
            ],
            nrows=0
        )
        .columns
        .tolist()
    )

    print(
        f"{name}:",
        feature,
        "->",
        feature in columns
    )

Combined National Context Source Files

CAS:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\CAS_Data_public.csv
Exists: True

Junction:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\spatial\CAS_Junction_enriched.csv
Exists: True

NZDep2023:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\nzdep2023\CAS_NZDep2023_enriched.csv
Exists: True

LCDB:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\lcdb\CAS_LCDB_v6_enriched.csv
Exists: True

Terrain:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\terrain\CAS_LINZ_terrain_enriched.csv
Exists: True

Required feature checks:
Junction: dist_to_junction_m -> True
NZDep2023: NZDep2023_Score -> True
LCDB: LCDB_Group_clean -> True
Terrain: Elevation_m -> True


In [2]:
# Load source datasets and validate crash IDs before merging.

import numpy as np

cas = pd.read_csv(CAS_PATH, low_memory=False)

# IDs correspond to the original CAS file's row order.
cas["cas_row_id"] = np.arange(len(cas), dtype=np.int64)

junction = pd.read_csv(
    JUNCTION_PATH,
    usecols=["cas_row_id", "dist_to_junction_m"]
)

nzdep = pd.read_csv(
    NZDEP_PATH,
    usecols=["cas_row_id", "NZDep2023_Score"]
)

lcdb = pd.read_csv(
    LCDB_PATH,
    usecols=["cas_row_id", "LCDB_Group_clean"]
)

terrain = pd.read_csv(
    TERRAIN_PATH,
    usecols=["cas_row_id", "Elevation_m"]
)

source_tables = {
    "CAS": cas,
    "Junction": junction,
    "NZDep2023": nzdep,
    "LCDB": lcdb,
    "Terrain": terrain
}

base_ids = set(cas["cas_row_id"])

print("Source Dataset Alignment Check")

for name, source_df in source_tables.items():
    ids = pd.to_numeric(
        source_df["cas_row_id"],
        errors="raise"
    )

    if ids.isna().any():
        raise ValueError(f"{name}: missing cas_row_id values.")

    if (
        not np.isfinite(ids.to_numpy()).all()
        or (ids < 0).any()
        or (ids % 1 != 0).any()
    ):
        raise ValueError(
            f"{name}: crash IDs must be finite, non-negative integers."
        )

    if ids.duplicated().any():
        raise ValueError(f"{name}: duplicate cas_row_id values.")

    source_ids = set(ids)
    missing_ids = base_ids - source_ids
    extra_ids = source_ids - base_ids

    if missing_ids or extra_ids:
        raise ValueError(
            f"{name}: crash IDs do not match CAS. "
            f"Missing IDs: {len(missing_ids)}; "
            f"extra IDs: {len(extra_ids)}."
        )

    source_df["cas_row_id"] = ids.astype(np.int64)

    print(
        f"{name}: {len(source_df):,} rows; "
        "unique IDs; complete CAS ID coverage."
    )

print("\nAll source ID checks passed.")

Source Dataset Alignment Check
CAS: 34,866 rows; unique IDs; complete CAS ID coverage.
Junction: 34,866 rows; unique IDs; complete CAS ID coverage.
NZDep2023: 34,866 rows; unique IDs; complete CAS ID coverage.
LCDB: 34,866 rows; unique IDs; complete CAS ID coverage.
Terrain: 34,866 rows; unique IDs; complete CAS ID coverage.

All source ID checks passed.


In [3]:
# Merge and validate before saving the combined dataset.

context_sources = {
    "Junction": (junction, "dist_to_junction_m"),
    "NZDep2023": (nzdep, "NZDep2023_Score"),
    "LCDB": (lcdb, "LCDB_Group_clean"),
    "Terrain": (terrain, "Elevation_m")
}

combined_context = cas.copy()
expected_ids = cas["cas_row_id"].to_numpy(copy=True)

for name, (source_df, feature) in context_sources.items():
    if feature in combined_context.columns:
        raise ValueError(
            f"{feature} already exists in CAS. "
            "Check the source file before merging."
        )

    combined_context = combined_context.merge(
        source_df[["cas_row_id", feature]],
        on="cas_row_id",
        how="left",
        sort=False,
        validate="one_to_one",
        indicator="_context_merge"
    )

    if not combined_context["_context_merge"].eq("both").all():
        raise ValueError(f"{name}: some CAS rows have no matching ID.")

    combined_context = combined_context.drop(
        columns="_context_merge"
    )

if len(combined_context) != len(cas):
    raise ValueError("The merge changed the number of CAS rows.")

if (
    combined_context["cas_row_id"].isna().any()
    or combined_context["cas_row_id"].duplicated().any()
):
    raise ValueError("Invalid crash IDs in the combined dataset.")

if not np.array_equal(
    combined_context["cas_row_id"].to_numpy(),
    expected_ids
):
    raise ValueError("The merge changed the original CAS row order.")

pd.testing.assert_frame_equal(
    combined_context[cas.columns].reset_index(drop=True),
    cas.reset_index(drop=True)
)

context_columns = [
    feature for _, feature in context_sources.values()
]

print("Combined National Context Dataset")
print(f"Rows: {len(combined_context):,}")
print("Columns:", len(combined_context.columns))
print("CAS row order and original columns preserved.")

# Missing feature values are distinct from missing source rows.
# Existing model pipelines handle missing values through imputation.
print("\nMissing contextual values:")
display(
    combined_context[context_columns]
    .isna()
    .sum()
    .rename("Missing_Count")
    .to_frame()
)

COMBINED_DIR = DATA_DIR / "combined_context"
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_PATH = (
    COMBINED_DIR / "CAS_National_Context_enriched.csv"
)

combined_context.to_csv(COMBINED_PATH, index=False)

print("\nValidation passed. Saved:")
print(COMBINED_PATH)

Combined National Context Dataset
Rows: 34,866
Columns: 77
CAS row order and original columns preserved.

Missing contextual values:


,Missing_Count
dist_to_junction_m,0
NZDep2023_Score,979
LCDB_Group_clean,0
Elevation_m,446



Validation passed. Saved:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\combined_context\CAS_National_Context_enriched.csv


In [4]:
# FINAL FEATURE DEFINITION
# Use the established CAS baseline from the earlier notebooks.

target_col = "crashSeverity"


# 1. Established CAS baseline: 30 predictors

baseline_predictors = [
    "bicycle",
    "bus",
    "carStationWagon",
    "crashDirectionDescription",
    "crashFinancialYear",
    "crashSHDescription",
    "crashYear",
    "directionRoleDescription",
    "flatHill",
    "light",
    "moped",
    "motorcycle",
    "NumberOfLanes",
    "otherVehicleType",
    "region",
    "roadCharacter",
    "roadLane",
    "roadSurface",
    "schoolBus",
    "speedLimit",
    "streetLight",
    "suv",
    "taxi",
    "trafficControl",
    "truck",
    "unknownVehicleType",
    "urban",
    "vanOrUtility",
    "weatherA",
    "weatherB"
]


# 2. Established categorical CAS predictors

baseline_categorical = [
    "crashDirectionDescription",
    "crashFinancialYear",
    "crashSHDescription",
    "directionRoleDescription",
    "flatHill",
    "light",
    "region",
    "roadCharacter",
    "roadLane",
    "roadSurface",
    "streetLight",
    "trafficControl",
    "urban",
    "weatherA",
    "weatherB"
]


# 3. Selected external contextual predictors

context_predictors = [
    "dist_to_junction_m",
    "NZDep2023_Score",
    "LCDB_Group_clean",
    "Elevation_m"
]

context_categorical = [
    "LCDB_Group_clean"
]


# 4. Final model: 30 CAS + 4 context predictors

final_predictors = (
    baseline_predictors
    + context_predictors
)

final_categorical = (
    baseline_categorical
    + context_categorical
)


# 5. Validate before training

if len(baseline_predictors) != 30:
    raise ValueError("Expected 30 CAS baseline predictors.")

if len(final_predictors) != 34:
    raise ValueError("Expected 34 final-model predictors.")

if len(set(final_predictors)) != len(final_predictors):
    raise ValueError("Duplicate predictor names.")

if not set(final_categorical).issubset(final_predictors):
    raise ValueError(
        "Categorical predictors must belong to the final feature set."
    )

required_columns = [
    target_col,
    *final_predictors
]

missing_features = [
    column
    for column in required_columns
    if column not in combined_context.columns
]

if missing_features:
    raise KeyError(
        f"Required columns are missing: {missing_features}"
    )

excluded_predictors = {
    "fatalCount",
    "seriousInjuryCount",
    "minorInjuryCount",
    "OBJECTID",
    "cas_row_id",
    "parkedVehicle",
    "advisorySpeed",
    "temporarySpeedLimit",
    "pedestrian",
    "holiday"
}

unexpected_predictors = (
    set(final_predictors)
    & excluded_predictors
)

if unexpected_predictors:
    raise ValueError(
        "Predictors excluded by the established baseline audit "
        f"were included: {sorted(unexpected_predictors)}"
    )


# 6. Display

print("FINAL FEATURE DEFINITION")

print(
    "\nCAS baseline predictors:",
    len(baseline_predictors)
)

print(
    "CAS categorical predictors:",
    len(baseline_categorical)
)

print(
    "CAS numeric predictors:",
    len(baseline_predictors) - len(baseline_categorical)
)

print(
    "\nAdded context predictors:",
    len(context_predictors)
)

print(
    "Final combined predictors:",
    len(final_predictors)
)

print(
    "Final categorical predictors:",
    len(final_categorical)
)

print(
    "Final numeric predictors:",
    len(final_predictors) - len(final_categorical)
)

print("\nSelected contextual predictors:")

for feature in context_predictors:
    print("-", feature)

print(
    "\nGCH included:",
    "GCH2023_clean" in final_predictors
)

print(
    "\nBaseline feature definitions now match "
    "the earlier experiment notebooks."
)

FINAL FEATURE DEFINITION

CAS baseline predictors: 30
CAS categorical predictors: 15
CAS numeric predictors: 15

Added context predictors: 4
Final combined predictors: 34
Final categorical predictors: 16
Final numeric predictors: 18

Selected contextual predictors:
- dist_to_junction_m
- NZDep2023_Score
- LCDB_Group_clean
- Elevation_m

GCH included: False

Baseline feature definitions now match the earlier experiment notebooks.


In [5]:
# Set up the four-class target and the five random seeds
# used for the paired robustness experiment.

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = (
    label_encoder
    .fit_transform(
        combined_context[
            target_col
        ]
    )
)

seeds = [
    42,
    7,
    21,
    84,
    123
]
print(
    "Target and Five-Seed Setup"
)


print(
    "\nTarget classes:"
)


for class_id, label in enumerate(
    label_encoder.classes_
):

    print(
        class_id,
        "=",
        label
    )


print(
    "\nNumber of crashes:",
    len(y_encoded)
)


print(
    "Five seeds:",
    seeds
)


print(
    "\nTarget distribution:"
)


target_distribution = (
    combined_context[
        target_col
    ]
    .value_counts()
    .reindex(
        label_encoder.classes_
    )
    .rename_axis(
        "Crash_Severity"
    )
    .reset_index(
        name="Count"
    )
)


target_distribution.insert(
    0,
    "Class_ID",
    range(
        len(
            target_distribution
        )
    )
)


display(
    target_distribution
)

Target and Five-Seed Setup

Target classes:
0 = Fatal Crash
1 = Minor Crash
2 = Non-Injury Crash
3 = Serious Crash

Number of crashes: 34866
Five seeds: [42, 7, 21, 84, 123]

Target distribution:


,Class_ID,Crash_Severity,Count
0,0,Fatal Crash,327
1,1,Minor Crash,9997
2,2,Non-Injury Crash,22036
3,3,Serious Crash,2506


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

from xgboost import XGBClassifier


def build_model_pipeline(
    predictors,
    categorical_cols,
    seed
):

    numeric_cols = [
        col
        for col in predictors
        if col not in categorical_cols
    ]


    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )


    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]
    )


    preprocess = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_cols
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_cols
            )
        ]
    )


    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=seed,
        n_jobs=-1
    )


    return Pipeline(
        steps=[
            (
                "preprocess",
                preprocess
            ),
            (
                "model",
                model
            )
        ]
    )


def run_seed_model(
    model_name,
    predictors,
    categorical_cols,
    train_idx,
    test_idx,
    seed
):

    X = (
        combined_context[
            predictors
        ]
        .copy()
    )


    X_train = X.iloc[
        train_idx
    ]

    X_test = X.iloc[
        test_idx
    ]


    y_train = y_encoded[
        train_idx
    ]

    y_test = y_encoded[
        test_idx
    ]


    sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train
    )


    pipeline = build_model_pipeline(
        predictors=predictors,
        categorical_cols=categorical_cols,
        seed=seed
    )


    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=
            sample_weights
    )


    y_pred = pipeline.predict(
        X_test
    )


    recalls = recall_score(
        y_test,
        y_pred,
        labels=np.arange(
            len(
                label_encoder.classes_
            )
        ),
        average=None,
        zero_division=0
    )


    recall_lookup = dict(
        zip(
            label_encoder.classes_,
            recalls
        )
    )


    return {
        "Seed":
            seed,

        "Model":
            model_name,

        "Accuracy":
            accuracy_score(
                y_test,
                y_pred
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_test,
                y_pred
            ),

        "Macro_F1":
            f1_score(
                y_test,
                y_pred,
                average="macro"
            ),

        "Fatal_Recall":
            recall_lookup[
                "Fatal Crash"
            ],

        "Serious_Recall":
            recall_lookup[
                "Serious Crash"
            ]
    }


print(
    "Model pipeline and evaluation function ready."
)

print(
    "CAS predictors:",
    len(baseline_predictors)
)

print(
    "Final context predictors:",
    len(final_predictors)
)

Model pipeline and evaluation function ready.
CAS predictors: 30
Final context predictors: 34


In [7]:
# Run paired five-seed evaluation using the exact same
# train/test split for both models within each seed.

paired_results = []


for seed in seeds:

    print(
        "\nRunning seed:",
        seed
    )


    train_idx, test_idx = train_test_split(
        np.arange(
            len(
                combined_context
            )
        ),
        test_size=0.20,
        random_state=seed,
        stratify=y_encoded
    )


    cas_result = run_seed_model(
        model_name="CAS Only",
        predictors=baseline_predictors,
        categorical_cols=baseline_categorical,
        train_idx=train_idx,
        test_idx=test_idx,
        seed=seed
    )


    final_result = run_seed_model(
        model_name="Final National Context",
        predictors=final_predictors,
        categorical_cols=final_categorical,
        train_idx=train_idx,
        test_idx=test_idx,
        seed=seed
    )


    paired_results.append(
        cas_result
    )

    paired_results.append(
        final_result
    )


paired_results_df = pd.DataFrame(
    paired_results
)


print(
    "\nFive-Seed Paired Results"
)


display(
    paired_results_df.round(4)
)


Running seed: 42

Running seed: 7

Running seed: 21

Running seed: 84

Running seed: 123

Five-Seed Paired Results


,Seed,Model,Accuracy,Balanced_Accuracy,Macro_F1,Fatal_Recall,Serious_Recall
0,42,CAS Only,0.4900,0.4076,0.3272,0.3231,0.3892
1,42,Final National Context,0.5206,0.4007,0.3440,0.2154,0.3952
2,7,CAS Only,0.4920,0.4111,0.3284,0.3385,0.3892
3,7,Final National Context,0.5106,0.3831,0.3322,0.2000,0.3633
4,21,CAS Only,0.4991,0.4217,0.3351,0.3538,0.4012
5,21,Final National Context,0.5212,0.4012,0.3406,0.2615,0.3613
6,84,CAS Only,0.4971,0.4024,0.3315,0.2615,0.4172
7,84,Final National Context,0.5218,0.3932,0.3398,0.2000,0.3812
8,123,CAS Only,0.4918,0.3881,0.3219,0.2615,0.3752
9,123,Final National Context,0.5175,0.3953,0.3379,0.2308,0.3772


In [8]:
# Summarise the five-seed results using mean and standard deviation
# and calculate paired gains relative to the CAS-only baseline.

metrics = [
    "Accuracy",
    "Balanced_Accuracy",
    "Macro_F1",
    "Fatal_Recall",
    "Serious_Recall"
]


summary_rows = []


for model_name in [
    "CAS Only",
    "Final National Context"
]:

    subset = (
        paired_results_df.loc[
            paired_results_df[
                "Model"
            ] == model_name
        ]
    )


    row = {
        "Model":
            model_name
    }


    for metric in metrics:

        row[
            metric + "_Mean"
        ] = (
            subset[
                metric
            ]
            .mean()
        )


        row[
            metric + "_SD"
        ] = (
            subset[
                metric
            ]
            .std()
        )


    summary_rows.append(
        row
    )


five_seed_summary = pd.DataFrame(
    summary_rows
)


print(
    "Five-Seed Mean ± SD Summary"
)


display(
    five_seed_summary.round(4)
)


paired_pivot = (
    paired_results_df
    .pivot(
        index="Seed",
        columns="Model",
        values=metrics
    )
)


gain_rows = []


for seed in seeds:

    row = {
        "Seed":
            seed
    }


    for metric in metrics:

        row[
            metric + "_Gain"
        ] = (
            paired_pivot.loc[
                seed,
                (
                    metric,
                    "Final National Context"
                )
            ]
            -
            paired_pivot.loc[
                seed,
                (
                    metric,
                    "CAS Only"
                )
            ]
        )


    gain_rows.append(
        row
    )


five_seed_gains = pd.DataFrame(
    gain_rows
)


print(
    "\nPaired Final Context - CAS Only Gains"
)


display(
    five_seed_gains.round(4)
)


print(
    "\nMean Paired Gains"
)


mean_gains = (
    five_seed_gains
    .drop(
        columns="Seed"
    )
    .mean()
    .to_frame(
        name="Mean_Gain"
    )
)


display(
    mean_gains.round(4)
)

Five-Seed Mean ± SD Summary


,Model,Accuracy_Mean,Accuracy_SD,Balanced_Accuracy_Mean,Balanced_Accuracy_SD,Macro_F1_Mean,Macro_F1_SD,Fatal_Recall_Mean,Fatal_Recall_SD,Serious_Recall_Mean,Serious_Recall_SD
0,CAS Only,0.4940,0.0039,0.4062,0.0123,0.3288,0.0049,0.3077,0.0435,0.3944,0.0157
1,Final National Context,0.5184,0.0046,0.3947,0.0073,0.3389,0.0043,0.2215,0.0257,0.3756,0.0139



Paired Final Context - CAS Only Gains


,Seed,Accuracy_Gain,Balanced_Accuracy_Gain,Macro_F1_Gain,Fatal_Recall_Gain,Serious_Recall_Gain
0,42,0.0307,-0.0069,0.0167,-0.1077,0.0060
1,7,0.0186,-0.0280,0.0038,-0.1385,-0.0259
2,21,0.0221,-0.0205,0.0055,-0.0923,-0.0399
3,84,0.0247,-0.0093,0.0082,-0.0615,-0.0359
4,123,0.0257,0.0072,0.0160,-0.0308,0.0020



Mean Paired Gains


,Mean_Gain
Accuracy_Gain,0.0243
Balanced_Accuracy_Gain,-0.0115
Macro_F1_Gain,0.0101
Fatal_Recall_Gain,-0.0862
Serious_Recall_Gain,-0.0188


In [9]:
# Save the reproducible five-seed experiment outputs.

PAIRED_RESULTS_PATH = (
    COMBINED_DIR
    / "final_five_seed_paired_results.csv"
)


SUMMARY_PATH = (
    COMBINED_DIR
    / "final_five_seed_summary.csv"
)


GAIN_PATH = (
    COMBINED_DIR
    / "final_five_seed_gain_summary.csv"
)


paired_results_df.to_csv(
    PAIRED_RESULTS_PATH,
    index=False
)


five_seed_summary.to_csv(
    SUMMARY_PATH,
    index=False
)


five_seed_gains.to_csv(
    GAIN_PATH,
    index=False
)


print(
    "Saved final experiment outputs:"
)


print(
    "\nPaired results:"
)

print(
    PAIRED_RESULTS_PATH
)


print(
    "\nFive-seed summary:"
)

print(
    SUMMARY_PATH
)


print(
    "\nPaired gain results:"
)

print(
    GAIN_PATH
)


print(
    "\nFinal mean Macro F1:"
)


for model_name in [
    "CAS Only",
    "Final National Context"
]:

    value = (
        five_seed_summary.loc[
            five_seed_summary[
                "Model"
            ] == model_name,
            "Macro_F1_Mean"
        ]
        .iloc[0]
    )

    print(
        model_name,
        "->",
        round(
            value,
            4
        )
    )


mean_macro_gain = (
    five_seed_gains[
        "Macro_F1_Gain"
    ]
    .mean()
)


print(
    "\nMean Macro F1 gain:",
    round(
        mean_macro_gain,
        4
    )
)


print(
    "Seeds with improved Macro F1:",
    int(
        (
            five_seed_gains[
                "Macro_F1_Gain"
            ] > 0
        )
        .sum()
    ),
    "/",
    len(seeds)
)

Saved final experiment outputs:

Paired results:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\combined_context\final_five_seed_paired_results.csv

Five-seed summary:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\combined_context\final_five_seed_summary.csv

Paired gain results:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\combined_context\final_five_seed_gain_summary.csv

Final mean Macro F1:
CAS Only -> 0.3288
Final National Context -> 0.3389

Mean Macro F1 gain: 0.0101
Seeds with improved Macro F1: 5 / 5


In [10]:
# Train the final national-context model for seed 42
# and save crash-level predictions for error analysis.

SEED = 42


train_idx, test_idx = train_test_split(
    np.arange(
        len(combined_context)
    ),
    test_size=0.20,
    random_state=SEED,
    stratify=y_encoded
)


X = (
    combined_context[
        final_predictors
    ]
    .copy()
)


X_train = X.iloc[
    train_idx
]

X_test = X.iloc[
    test_idx
]


y_train = y_encoded[
    train_idx
]

y_test = y_encoded[
    test_idx
]


sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)


final_seed42_pipeline = build_model_pipeline(
    predictors=final_predictors,
    categorical_cols=final_categorical,
    seed=SEED
)


final_seed42_pipeline.fit(
    X_train,
    y_train,
    model__sample_weight=sample_weights
)


y_pred = final_seed42_pipeline.predict(
    X_test
)


y_proba = final_seed42_pipeline.predict_proba(
    X_test
)


prediction_confidence = y_proba.max(
    axis=1
)


true_class_probability = y_proba[
    np.arange(
        len(y_test)
    ),
    y_test
]


actual_labels = label_encoder.inverse_transform(
    y_test
)


predicted_labels = label_encoder.inverse_transform(
    y_pred
)


severity_rank = {
    "Non-Injury Crash": 0,
    "Minor Crash": 1,
    "Serious Crash": 2,
    "Fatal Crash": 3
}


error_direction = []


for actual, predicted in zip(
    actual_labels,
    predicted_labels
):

    if actual == predicted:
        direction = "Correct"

    elif (
        severity_rank[predicted]
        <
        severity_rank[actual]
    ):
        direction = "Under-classified"

    else:
        direction = "Over-classified"

    error_direction.append(
        direction
    )


error_analysis = (
    combined_context
    .iloc[
        test_idx
    ]
    .copy()
)


error_analysis[
    "Actual_Class_ID"
] = y_test


error_analysis[
    "Predicted_Class_ID"
] = y_pred


error_analysis[
    "Actual_Severity"
] = actual_labels


error_analysis[
    "Predicted_Severity"
] = predicted_labels


error_analysis[
    "Actual"
] = actual_labels


error_analysis[
    "Predicted"
] = predicted_labels


error_analysis[
    "Prediction_Confidence"
] = prediction_confidence


error_analysis[
    "True_Class_Probability"
] = true_class_probability


error_analysis[
    "Error_Direction"
] = error_direction


error_analysis[
    "Correct_Prediction"
] = (
    y_test
    ==
    y_pred
)


ERROR_ANALYSIS_PATH = (
    COMBINED_DIR
    / "final_model_error_analysis_seed42.csv"
)


error_analysis.to_csv(
    ERROR_ANALYSIS_PATH,
    index=False
)


print(
    "Seed-42 Final Model Error Analysis"
)


print(
    "\nTest crashes:",
    len(error_analysis)
)


print(
    "Correct predictions:",
    int(
        error_analysis[
            "Correct_Prediction"
        ].sum()
    )
)


print(
    "Incorrect predictions:",
    int(
        (
            ~error_analysis[
                "Correct_Prediction"
            ]
        ).sum()
    )
)


print(
    "\nConfusion table:"
)


confusion_table = pd.crosstab(
    error_analysis[
        "Actual"
    ],
    error_analysis[
        "Predicted"
    ],
    margins=True
)


display(
    confusion_table
)


print(
    "\nSaved error-analysis file:"
)

print(
    ERROR_ANALYSIS_PATH
)

Seed-42 Final Model Error Analysis

Test crashes: 6974
Correct predictions: 3631
Incorrect predictions: 3343

Confusion table:


Predicted,Fatal Crash,Minor Crash,Non-Injury Crash,Serious Crash,All
Actual,,,,,
Fatal Crash,14,11,10,30,65
Minor Crash,113,793,698,396,2000
Non-Injury Crash,206,1165,2626,411,4408
Serious Crash,31,172,100,198,501
All,364,2141,3434,1035,6974



Saved error-analysis file:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\combined_context\final_model_error_analysis_seed42.csv


In [11]:
# Summarise prediction outcomes for Fatal and Serious crashes.

severe_classes = [
    "Fatal Crash",
    "Serious Crash"
]


severe_error_summary = (
    error_analysis.loc[
        error_analysis[
            "Actual_Severity"
        ].isin(
            severe_classes
        )
    ]
    .groupby(
        [
            "Actual_Severity",
            "Predicted_Severity"
        ],
        observed=False
    )
    .size()
    .reset_index(
        name="Count"
    )
)


severe_totals = (
    severe_error_summary
    .groupby(
        "Actual_Severity",
        observed=False
    )[
        "Count"
    ]
    .transform(
        "sum"
    )
)


severe_error_summary[
    "Percentage"
] = (
    severe_error_summary[
        "Count"
    ]
    / severe_totals
    * 100
)


severity_order = [
    "Fatal Crash",
    "Serious Crash"
]


prediction_order = [
    "Fatal Crash",
    "Serious Crash",
    "Minor Crash",
    "Non-Injury Crash"
]


severe_error_summary[
    "Actual_Severity"
] = pd.Categorical(
    severe_error_summary[
        "Actual_Severity"
    ],
    categories=severity_order,
    ordered=True
)


severe_error_summary[
    "Predicted_Severity"
] = pd.Categorical(
    severe_error_summary[
        "Predicted_Severity"
    ],
    categories=prediction_order,
    ordered=True
)


severe_error_summary = (
    severe_error_summary
    .sort_values(
        [
            "Actual_Severity",
            "Predicted_Severity"
        ]
    )
    .reset_index(
        drop=True
    )
)


fatal_total = (
    error_analysis[
        "Actual_Severity"
    ]
    .eq(
        "Fatal Crash"
    )
    .sum()
)


fatal_correct = (
    (
        error_analysis[
            "Actual_Severity"
        ]
        .eq(
            "Fatal Crash"
        )
    )
    &
    (
        error_analysis[
            "Predicted_Severity"
        ]
        .eq(
            "Fatal Crash"
        )
    )
).sum()


serious_total = (
    error_analysis[
        "Actual_Severity"
    ]
    .eq(
        "Serious Crash"
    )
    .sum()
)


serious_correct = (
    (
        error_analysis[
            "Actual_Severity"
        ]
        .eq(
            "Serious Crash"
        )
    )
    &
    (
        error_analysis[
            "Predicted_Severity"
        ]
        .eq(
            "Serious Crash"
        )
    )
).sum()


fatal_as_serious = (
    (
        error_analysis[
            "Actual_Severity"
        ]
        .eq(
            "Fatal Crash"
        )
    )
    &
    (
        error_analysis[
            "Predicted_Severity"
        ]
        .eq(
            "Serious Crash"
        )
    )
).sum()


serious_downgraded = (
    (
        error_analysis[
            "Actual_Severity"
        ]
        .eq(
            "Serious Crash"
        )
    )
    &
    (
        error_analysis[
            "Predicted_Severity"
        ]
        .isin(
            [
                "Minor Crash",
                "Non-Injury Crash"
            ]
        )
    )
).sum()


fatal_recall_seed42 = (
    fatal_correct
    / fatal_total
)


serious_recall_seed42 = (
    serious_correct
    / serious_total
)


fatal_as_serious_pct = (
    fatal_as_serious
    / fatal_total
    * 100
)


serious_downgraded_pct = (
    serious_downgraded
    / serious_total
    * 100
)


print(
    "Fatal and Serious Crash Error Summary"
)


display(
    severe_error_summary.round(
        {
            "Percentage": 2
        }
    )
)


print(
    "\nFatal crash recall:",
    round(
        fatal_recall_seed42,
        4
    )
)


print(
    "Serious crash recall:",
    round(
        serious_recall_seed42,
        4
    )
)


print(
    "\nFatal crashes predicted as Serious:",
    round(
        fatal_as_serious_pct,
        2
    ),
    "%"
)


print(
    "Serious crashes predicted as Minor or Non-Injury:",
    round(
        serious_downgraded_pct,
        2
    ),
    "%"
)

Fatal and Serious Crash Error Summary


,Actual_Severity,Predicted_Severity,Count,Percentage
0,Fatal Crash,Fatal Crash,14,21.54
1,Fatal Crash,Serious Crash,30,46.15
2,Fatal Crash,Minor Crash,11,16.92
3,Fatal Crash,Non-Injury Crash,10,15.38
4,Serious Crash,Fatal Crash,31,6.19
5,Serious Crash,Serious Crash,198,39.52
6,Serious Crash,Minor Crash,172,34.33
7,Serious Crash,Non-Injury Crash,100,19.96



Fatal crash recall: 0.2154
Serious crash recall: 0.3952

Fatal crashes predicted as Serious: 46.15 %
Serious crashes predicted as Minor or Non-Injury: 54.29 %


In [12]:
# Build a concise interpretation summary from the
# reproduced five-seed and severe-class results.

cas_summary = (
    five_seed_summary.loc[
        five_seed_summary[
            "Model"
        ] == "CAS Only"
    ]
    .iloc[0]
)


final_summary = (
    five_seed_summary.loc[
        five_seed_summary[
            "Model"
        ] == "Final National Context"
    ]
    .iloc[0]
)


macro_f1_improved_seeds = (
    five_seed_gains[
        "Macro_F1_Gain"
    ] > 0
).sum()


interpretation_rows = [
    {
        "Finding":
            "Accuracy",

        "CAS_Only":
            cas_summary[
                "Accuracy_Mean"
            ],

        "Final_Context":
            final_summary[
                "Accuracy_Mean"
            ],

        "Change":
            final_summary[
                "Accuracy_Mean"
            ]
            -
            cas_summary[
                "Accuracy_Mean"
            ],

        "Interpretation":
            (
                "The final national-context model "
                "improved overall classification accuracy."
            )
    },

    {
        "Finding":
            "Balanced Accuracy",

        "CAS_Only":
            cas_summary[
                "Balanced_Accuracy_Mean"
            ],

        "Final_Context":
            final_summary[
                "Balanced_Accuracy_Mean"
            ],

        "Change":
            final_summary[
                "Balanced_Accuracy_Mean"
            ]
            -
            cas_summary[
                "Balanced_Accuracy_Mean"
            ],

        "Interpretation":
            (
                "Balanced accuracy decreased, indicating "
                "weaker average class-level performance."
            )
    },

    {
        "Finding":
            "Macro F1",

        "CAS_Only":
            cas_summary[
                "Macro_F1_Mean"
            ],

        "Final_Context":
            final_summary[
                "Macro_F1_Mean"
            ],

        "Change":
            final_summary[
                "Macro_F1_Mean"
            ]
            -
            cas_summary[
                "Macro_F1_Mean"
            ],

        "Interpretation":
            (
                "Macro F1 improved in all five seeds, "
                "showing a consistent overall benefit "
                "from the added national-context features."
            )
    },

    {
        "Finding":
            "Fatal Recall",

        "CAS_Only":
            cas_summary[
                "Fatal_Recall_Mean"
            ],

        "Final_Context":
            final_summary[
                "Fatal_Recall_Mean"
            ],

        "Change":
            final_summary[
                "Fatal_Recall_Mean"
            ]
            -
            cas_summary[
                "Fatal_Recall_Mean"
            ],

        "Interpretation":
            (
                "Fatal-crash recall decreased substantially. "
                "The final model often confused Fatal crashes "
                "with Serious crashes."
            )
    },

    {
        "Finding":
            "Serious Recall",

        "CAS_Only":
            cas_summary[
                "Serious_Recall_Mean"
            ],

        "Final_Context":
            final_summary[
                "Serious_Recall_Mean"
            ],

        "Change":
            final_summary[
                "Serious_Recall_Mean"
            ]
            -
            cas_summary[
                "Serious_Recall_Mean"
            ],

        "Interpretation":
            (
                "Serious-crash recall changed only slightly "
                "across the five-seed comparison."
            )
    }
]


interpretation_summary = pd.DataFrame(
    interpretation_rows
)


INTERPRETATION_PATH = (
    COMBINED_DIR
    / "final_combined_interpretation_summary.csv"
)


interpretation_summary.to_csv(
    INTERPRETATION_PATH,
    index=False
)


print(
    "Final Combined National Context Interpretation"
)


display(
    interpretation_summary.round(
        {
            "CAS_Only": 4,
            "Final_Context": 4,
            "Change": 4
        }
    )
)


print(
    "\nAdditional severe-class findings:"
)


print(
    "Fatal crashes predicted as Serious:",
    f"{fatal_as_serious_pct:.2f}%"
)


print(
    "Serious crashes predicted as Minor or Non-Injury:",
    f"{serious_downgraded_pct:.2f}%"
)


print(
    "Macro F1 improved in:",
    int(
        macro_f1_improved_seeds
    ),
    "/",
    len(seeds),
    "seeds"
)


print(
    "\nSaved interpretation summary:"
)

print(
    INTERPRETATION_PATH
)

Final Combined National Context Interpretation


,Finding,CAS_Only,Final_Context,Change,Interpretation
0,Accuracy,0.4940,0.5184,0.0243,The final national-context model improved over...
1,Balanced Accuracy,0.4062,0.3947,-0.0115,"Balanced accuracy decreased, indicating weaker..."
2,Macro F1,0.3288,0.3389,0.0101,"Macro F1 improved in all five seeds, showing a..."
3,Fatal Recall,0.3077,0.2215,-0.0862,Fatal-crash recall decreased substantially. Th...
4,Serious Recall,0.3944,0.3756,-0.0188,Serious-crash recall changed only slightly acr...



Additional severe-class findings:
Fatal crashes predicted as Serious: 46.15%
Serious crashes predicted as Minor or Non-Injury: 54.29%
Macro F1 improved in: 5 / 5 seeds

Saved interpretation summary:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\combined_context\final_combined_interpretation_summary.csv


In [13]:
# REGENERATE NATIONAL SHAP RESULTS
# Latest seed-42 final model
# Uses the complete held-out test set.

import numpy as np
import pandas as pd
import xgboost as xgb


# 1. Check required fitted objects

required_variables = [
    "final_seed42_pipeline",
    "X_test",
    "label_encoder",
    "COMBINED_DIR"
]

missing = [
    name
    for name in required_variables
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run all existing Notebook 08 cells first. "
        f"Missing variables: {missing}"
    )

preprocessor = final_seed42_pipeline.named_steps["preprocess"]
model = final_seed42_pipeline.named_steps["model"]

X_transformed = preprocessor.transform(X_test)

booster = model.get_booster()
test_matrix = xgb.DMatrix(X_transformed)


# 2. Calculate exact TreeSHAP contributions
# Shape: test rows × classes × (transformed features + bias)
# Contributions explain raw class margins, not probabilities.

contributions = booster.predict(
    test_matrix,
    pred_contribs=True,
    approx_contribs=False,
    strict_shape=True
)

margins = booster.predict(
    test_matrix,
    output_margin=True,
    strict_shape=True
)

np.testing.assert_allclose(
    contributions.sum(axis=2),
    margins,
    rtol=1e-4,
    atol=1e-4
)

shap_values = contributions[:, :, :-1]

class_names = label_encoder.inverse_transform(
    model.classes_.astype(int)
)

assert shap_values.shape[1] == len(class_names)


# 3. Map transformed columns to original predictors

numeric_columns = list(
    next(
        columns
        for name, transformer, columns
        in preprocessor.transformers_
        if name == "numeric"
    )
)

categorical_columns = list(
    next(
        columns
        for name, transformer, columns
        in preprocessor.transformers_
        if name == "categorical"
    )
)

numeric_slice = preprocessor.output_indices_["numeric"]
categorical_slice = preprocessor.output_indices_["categorical"]

numeric_width = numeric_slice.stop - numeric_slice.start

if numeric_width != len(numeric_columns):
    raise ValueError(
        "Numeric preprocessing changed the number of columns. "
        "Check for entirely missing numeric predictors."
    )

feature_groups = {
    column: [numeric_slice.start + position]
    for position, column in enumerate(numeric_columns)
}

onehot = (
    preprocessor.named_transformers_["categorical"]
    .named_steps["onehot"]
)

offset = categorical_slice.start

for column, categories in zip(
    categorical_columns,
    onehot.categories_
):
    width = len(categories)

    feature_groups[column] = list(
        range(offset, offset + width)
    )

    offset += width

if offset != categorical_slice.stop:
    raise ValueError(
        "Categorical encoding does not match the expected columns."
    )


# 4. Sum signed SHAP contributions within each original predictor
# Then calculate mean absolute importance.

predictor_names = list(feature_groups)

grouped_shap = np.stack(
    [
        shap_values[:, :, feature_groups[column]].sum(axis=2)
        for column in predictor_names
    ],
    axis=2
)

np.testing.assert_allclose(
    grouped_shap.sum(axis=2),
    shap_values.sum(axis=2),
    rtol=1e-4,
    atol=1e-4
)


# 5. Global importance: average equally across test rows and classes

global_shap_summary = pd.DataFrame({
    "Feature": predictor_names,
    "Mean_Absolute_SHAP": np.abs(grouped_shap).mean(
        axis=(0, 1)
    )
})

global_shap_summary = (
    global_shap_summary
    .sort_values("Mean_Absolute_SHAP", ascending=False)
    .reset_index(drop=True)
)

global_shap_summary.insert(
    0,
    "Rank",
    np.arange(1, len(global_shap_summary) + 1)
)


# 6. Class-specific importance
# Each class is explained over the same complete test set.

class_tables = []

for class_position, class_name in enumerate(class_names):

    values = grouped_shap[:, class_position, :]

    table = pd.DataFrame({
        "Class": class_name,
        "Feature": predictor_names,
        "Mean_Absolute_SHAP": np.abs(values).mean(axis=0),
        "Mean_Signed_SHAP": values.mean(axis=0)
    })

    table = (
        table
        .sort_values("Mean_Absolute_SHAP", ascending=False)
        .reset_index(drop=True)
    )

    table.insert(
        1,
        "Rank",
        np.arange(1, len(table) + 1)
    )

    class_tables.append(table)

class_shap_summary = pd.concat(
    class_tables,
    ignore_index=True
)


# 7. Save reproducible outputs

SHAP_DIR = COMBINED_DIR / "shap_seed42"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

GLOBAL_PATH = SHAP_DIR / "national_shap_global_seed42.csv"
CLASS_PATH = SHAP_DIR / "national_shap_by_class_seed42.csv"

global_shap_summary.to_csv(
    GLOBAL_PATH,
    index=False
)

class_shap_summary.to_csv(
    CLASS_PATH,
    index=False
)

np.savez_compressed(
    SHAP_DIR / "national_shap_grouped_seed42.npz",
    values=grouped_shap,
    features=np.asarray(predictor_names, dtype=str),
    classes=np.asarray(class_names, dtype=str),
    test_index=X_test.index.to_numpy()
)


# 8. Display

print("Explained test crashes:", len(X_test))
print("SHAP additivity checks passed.")

print("\nGLOBAL SHAP IMPORTANCE")
display(global_shap_summary.head(15))

for class_name in ["Fatal Crash", "Serious Crash"]:
    print(f"\n{class_name.upper()} — SHAP IMPORTANCE")

    display(
        class_shap_summary.loc[
            class_shap_summary["Class"].eq(class_name)
        ].head(15)
    )

print("\nSaved:")
print(GLOBAL_PATH)
print(CLASS_PATH)

print(
    "\nSHAP values explain raw class margins. "
    "Mean signed contributions alone do not establish "
    "a monotonic relationship or causal effect."
)

Explained test crashes: 6974
SHAP additivity checks passed.

GLOBAL SHAP IMPORTANCE


,Rank,Feature,Mean_Absolute_SHAP
0,1,dist_to_junction_m,0.135356
1,2,motorcycle,0.112051
2,3,LCDB_Group_clean,0.110596
3,4,NZDep2023_Score,0.104752
4,5,speedLimit,0.103150
5,6,carStationWagon,0.099278
6,7,light,0.090584
7,8,roadLane,0.088530
8,9,Elevation_m,0.084670
9,10,streetLight,0.081191



FATAL CRASH — SHAP IMPORTANCE


,Class,Rank,Feature,Mean_Absolute_SHAP,Mean_Signed_SHAP
0,Fatal Crash,1,speedLimit,0.301204,-0.161414
1,Fatal Crash,2,dist_to_junction_m,0.285452,-0.256926
2,Fatal Crash,3,NZDep2023_Score,0.272411,-0.225997
3,Fatal Crash,4,Elevation_m,0.202134,-0.165644
4,Fatal Crash,5,roadLane,0.193506,-0.097662
5,Fatal Crash,6,light,0.159350,-0.066198
6,Fatal Crash,7,LCDB_Group_clean,0.151213,-0.072875
7,Fatal Crash,8,region,0.142887,-0.094856
8,Fatal Crash,9,crashDirectionDescription,0.104176,-0.075031
9,Fatal Crash,10,truck,0.100877,-0.036669



SERIOUS CRASH — SHAP IMPORTANCE


,Class,Rank,Feature,Mean_Absolute_SHAP,Mean_Signed_SHAP
102,Serious Crash,1,carStationWagon,0.179154,-0.085883
103,Serious Crash,2,flatHill,0.114446,-0.090402
104,Serious Crash,3,motorcycle,0.110575,-0.056335
105,Serious Crash,4,streetLight,0.100256,-0.061486
106,Serious Crash,5,dist_to_junction_m,0.088433,-0.025065
107,Serious Crash,6,NZDep2023_Score,0.071346,-0.007188
108,Serious Crash,7,weatherA,0.066494,-0.009470
109,Serious Crash,8,Elevation_m,0.058660,-0.011402
110,Serious Crash,9,light,0.050960,-0.004378
111,Serious Crash,10,region,0.048484,0.000147



Saved:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\combined_context\shap_seed42\national_shap_global_seed42.csv
C:\Users\ASUS\Desktop\Project_Roadcrash\data\combined_context\shap_seed42\national_shap_by_class_seed42.csv

SHAP values explain raw class margins. Mean signed contributions alone do not establish a monotonic relationship or causal effect.


In [14]:
# CURRENT FINAL MODEL: LEAVE-ONE-CONTEXT-OUT ABLATION
# Seed 42, identical train/test split for every configuration
# Four contextual predictors; GCH is not part of this experiment.

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


# 1. Check required objects

required_variables = [
    "combined_context",
    "baseline_predictors",
    "final_predictors",
    "final_categorical",
    "y_encoded",
    "run_seed_model",
    "COMBINED_DIR"
]

missing = [
    name
    for name in required_variables
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run the existing Notebook 08 cells first. "
        f"Missing variables: {missing}"
    )


# 2. Identify the four added contextual predictors

context_predictors = [
    feature
    for feature in final_predictors
    if feature not in baseline_predictors
]

expected_context = {
    "dist_to_junction_m",
    "NZDep2023_Score",
    "LCDB_Group_clean",
    "Elevation_m"
}

if set(context_predictors) != expected_context:
    raise ValueError(
        "The contextual predictors differ from the expected final model. "
        f"Found: {context_predictors}"
    )

if not set(baseline_predictors).issubset(final_predictors):
    raise ValueError(
        "The final model must retain all baseline predictors "
        "for this leave-one-context-out comparison."
    )


# 3. Reproduce the paired seed-42 split

ABLATION_SEED = 42

ablation_train_idx, ablation_test_idx = train_test_split(
    np.arange(len(combined_context)),
    test_size=0.20,
    random_state=ABLATION_SEED,
    stratify=y_encoded
)


# 4. Evaluate the complete final model

full_result = run_seed_model(
    model_name="Full Final National Context",
    predictors=list(final_predictors),
    categorical_cols=list(final_categorical),
    train_idx=ablation_train_idx,
    test_idx=ablation_test_idx,
    seed=ABLATION_SEED
)

full_macro_f1 = full_result["Macro_F1"]

ablation_rows = [{
    **full_result,
    "Removed_Feature": "None",
    "Number_of_Predictors": len(final_predictors),
    "Macro_F1_Loss_When_Removed": 0.0
}]


# 5. Remove one contextual predictor at a time

for removed_feature in context_predictors:

    print("\nRemoving:", removed_feature)

    reduced_predictors = [
        feature
        for feature in final_predictors
        if feature != removed_feature
    ]

    reduced_categorical = [
        feature
        for feature in final_categorical
        if feature in reduced_predictors
    ]

    result = run_seed_model(
        model_name=f"Final Context without {removed_feature}",
        predictors=reduced_predictors,
        categorical_cols=reduced_categorical,
        train_idx=ablation_train_idx,
        test_idx=ablation_test_idx,
        seed=ABLATION_SEED
    )

    ablation_rows.append({
        **result,
        "Removed_Feature": removed_feature,
        "Number_of_Predictors": len(reduced_predictors),

        # Positive: removal reduced Macro F1.
        # Negative: removal improved Macro F1.
        "Macro_F1_Loss_When_Removed": (
            full_macro_f1 - result["Macro_F1"]
        )
    })


# 6. Save the calculated results

current_ablation_results = pd.DataFrame(ablation_rows)

ABLATION_PATH = (
    COMBINED_DIR
    / "final_context_ablation_seed42.csv"
)

current_ablation_results.to_csv(
    ABLATION_PATH,
    index=False
)


# 7. Display

display_columns = [
    "Removed_Feature",
    "Number_of_Predictors",
    "Accuracy",
    "Balanced_Accuracy",
    "Macro_F1",
    "Fatal_Recall",
    "Serious_Recall",
    "Macro_F1_Loss_When_Removed"
]

print("\nCURRENT FINAL MODEL: SEED-42 ABLATION")

display(
    current_ablation_results[
        display_columns
    ].round(4)
)

print("\nSaved:")
print(ABLATION_PATH)

print(
    "\nPositive loss means removing the feature reduced Macro F1. "
    "Negative loss means removing it improved Macro F1. "
    "This is a single-split ablation and does not establish "
    "multi-seed robustness or causal importance."
)


Removing: dist_to_junction_m

Removing: NZDep2023_Score

Removing: LCDB_Group_clean

Removing: Elevation_m

CURRENT FINAL MODEL: SEED-42 ABLATION


,Removed_Feature,Number_of_Predictors,Accuracy,Balanced_Accuracy,Macro_F1,Fatal_Recall,Serious_Recall,Macro_F1_Loss_When_Removed
0,None,34,0.5206,0.4007,0.3440,0.2154,0.3952,0.0000
1,dist_to_junction_m,33,0.5113,0.4025,0.3400,0.2308,0.4092,0.0040
2,NZDep2023_Score,33,0.5122,0.3961,0.3372,0.2308,0.3772,0.0068
3,LCDB_Group_clean,33,0.5136,0.3951,0.3399,0.2000,0.3952,0.0040
4,Elevation_m,33,0.5145,0.4043,0.3409,0.2462,0.3912,0.0030



Saved:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\combined_context\final_context_ablation_seed42.csv

Positive loss means removing the feature reduced Macro F1. Negative loss means removing it improved Macro F1. This is a single-split ablation and does not establish multi-seed robustness or causal importance.
